In [ ]:
import json
import os

import dotenv
from loguru import logger
from models import Settings
from neo4j import GraphDatabase
from openai import OpenAI

dotenv.load_dotenv()

DATASET = "../data/ocre20251021.json"
settings = Settings()

In [ ]:
class CoinsIngestor:
    def __init__(self, dataset):
        self._dataset = dataset
        self._driver = GraphDatabase.driver(
            settings.neo4j_uri, auth=(settings.neo4j_username, settings.neo4j_password)
        )
        self._session = self._driver.session(database=settings.neo4j_database)
        self._openai = OpenAI(api_key=settings.openai_api_key)
        print(settings)
        self._create_index()

    def ingest(self):
        with open(self._dataset, "r") as f:
            data = json.load(f)

        for record in data:
            self._create_coins(record)

    def _create_coins(self, record):
        record_id = record.get("record_id")
        title = record.get("title", "")
        description = record.get("description", "")
        object_type = record.get("object_type", "")
        from_year = record.get("from_year")
        to_year = record.get("to_year")
        date_range = record.get("date_range", "")
        text_to_embed = f"{title}\n{description}"
        embedding_response = self._openai.embeddings.create(
            model=settings.embedding_model, input=text_to_embed
        )
        embedding = embedding_response.data[0].embedding

        self._session.run(
            """
            MERGE (c:Coin {record_id: $record_id})
            SET c.title = $title,
                c.description = $description,
                c.object_type = $object_type,
                c.from_year = $from_year,
                c.to_year = $to_year,
                c.date_range = $date_range,
                c.embedding = $embedding
        """,
            {
                "record_id": record_id,
                "title": title,
                "description": description,
                "object_type": object_type,
                "from_year": from_year,
                "to_year": to_year,
                "date_range": date_range,
                "embedding": embedding,
            },
        )

        authority = record.get("authority", [])
        for king in authority:
            if not king:
                continue

            self._session.run(
                """
                MERGE (a:Authority {name: $name})
                MERGE (c:Coin {record_id: $record_id})
                MERGE (c)-[:HAS_AUTHORITY]->(a)
                """,
                {"name": king, "record_id": record_id},
            )

        denomination = record.get("denomination", [])
        for denom in denomination:
            if not denom:
                continue

            self._session.run(
                """
                MERGE (d:Denomination {name: $name})
                MERGE (c:Coin {record_id: $record_id})
                MERGE (c)-[:HAS_DENOMINATION]->(d)
                """,
                {"name": denom, "record_id": record_id},
            )

        manufacturer = record.get("manufacturer", [])
        for method in manufacturer:
            if not method:
                continue

            self._session.run(
                """
                MERGE (m:Manufacturer {method: $method})
                MERGE (c:Coin {record_id: $record_id})
                MERGE (c)-[:MANUFACTURED_BY]->(m)
                """,
                {"method": method, "record_id": record_id},
            )

        materials = record.get("material", [])
        for material in materials:
            if not material:
                continue

            self._session.run(
                """
                MERGE (mat:Material {name: $name})
                MERGE (c:Coin {record_id: $record_id})
                MERGE (c)-[:MADE_OF]->(mat)
                """,
                {"name": material, "record_id": record_id},
            )

        geographic = record.get("geographic", [])
        for place in geographic:
            name = place.get("name")
            type = place.get("type")
            if not name and not type:
                continue

            self._session.run(
                """
                MERGE (p:Place {name: $name, type: $type})
                MERGE (c:Coin {record_id: $record_id})
                MERGE (c)-[:MINTED_AT]->(p)
                """,
                {"name": name, "type": type, "record_id": record_id},
            )

        logger.info(f"Created/Updated Coin node with record_id: {record_id}")

    def _create_index(self):
        self._session.run("""
            CREATE INDEX coin_record_id_index IF NOT EXISTS
            FOR (c:Coin)
            ON (c.record_id)
        """)
        logger.info("Created index on :Coin(record_id)")

        self._session.run("""
            CREATE FULLTEXT INDEX coins_text_index IF NOT EXISTS
            FOR (c:Coin)
            ON EACH [c.title, c.description]
        """)
        logger.info("Created fulltext index on :Coin(title, description)")

        self._session.run(f"""
            CREATE VECTOR INDEX coins_embeddings_index IF NOT EXISTS
            FOR (c:Coin)
            ON (c.embedding)
            OPTIONS {{ indexConfig: {{
                `vector.dimensions`: {settings.embedding_dimension},
                `vector.similarity_function`: 'cosine'
            }} }}
        """)
        logger.info("Created vector index on :Coin(embedding)")


ingestor = CoinsIngestor(DATASET)
ingestor.ingest()